# Spam Dataset Feature Engineering (Colab Fallback)

This notebook provides the exact same functionality as `src/spam/features/build_features.py` but is optimized to be run in an environment with more RAM, such as Google Colab. If your local machine crashes with Out of Memory errors, upload this notebook and the `data/spam/raw` directory to Colab, run it, and download the output `combined_data.csv`, `tfidf_matrix.npz`, and `tfidf_vectorizer.pkl` back to your local `data/spam/processed/` and `models/spam/` directories.

In [ ]:
import os
import sys
import csv
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import joblib

maxInt = sys.maxsize
while True:
    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/10)

# Set these paths appropriately for your environment (e.g. '/content/drive/MyDrive/...' if on Colab)
RAW_DIR = '../../data/spam/raw'
PROCESSED_DIR = '../../data/spam/processed'
MODELS_DIR = '../../models/spam'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)


In [ ]:
print("Loading datasets...")
df1_path = os.path.join(RAW_DIR, "emails.csv")
df1 = pd.DataFrame()
if os.path.exists(df1_path):
    df1 = pd.read_csv(df1_path)
    df1 = df1[["text", "spam"]]
    print(f"emails.csv loaded: {len(df1)} rows.")

df2_path = os.path.join(RAW_DIR, "enron_spam_data.csv")
df2 = pd.DataFrame()
if os.path.exists(df2_path):
    df2 = pd.read_csv(df2_path)
    df2 = df2.rename(columns={'Message': 'text', 'Spam/Ham': 'spam'})
    df2['spam'] = df2['spam'].map({'spam': 1, 'ham': 0})
    df2 = df2[["text", "spam"]]
    print(f"enron_spam_data.csv loaded: {len(df2)} rows.")

df3_path = os.path.join(RAW_DIR, "processed_data.csv")
df3 = pd.DataFrame()
if os.path.exists(df3_path):
    df3 = pd.read_csv(df3_path, engine='python', on_bad_lines='skip')
    df3 = df3.rename(columns={'message': 'text', 'label': 'spam'})
    df3 = df3[["text", "spam"]]
    print(f"processed_data.csv loaded: {len(df3)} rows.")

print("Combining datasets...")
combined_df = pd.concat([df1, df2, df3], ignore_index=True)

initial_len = len(combined_df)
combined_df = combined_df.dropna(subset=['text', 'spam'])
combined_df = combined_df.drop_duplicates(subset=['text'])
print(f"Combined dataset: {len(combined_df)} rows (dropped {initial_len - len(combined_df)} NAs/duplicates).")


In [ ]:
combined_path = os.path.join(PROCESSED_DIR, "combined_data.csv")
print(f"Saving combined dataset to {combined_path}...")
combined_df.to_csv(combined_path, index=False)

print("Extracting TF-IDF features...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(combined_df['text'])
y = combined_df['spam'].values

print(f"TF-IDF matrix shape: {X.shape}")

matrix_path = os.path.join(PROCESSED_DIR, "tfidf_matrix.npz")
print(f"Saving TF-IDF matrix to {matrix_path}...")
scipy.sparse.save_npz(matrix_path, X)

labels_path = os.path.join(PROCESSED_DIR, "labels.npy")
print(f"Saving labels to {labels_path}...")
np.save(labels_path, y)

vectorizer_path = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
print(f"Saving vectorizer to {vectorizer_path}...")
joblib.dump(vectorizer, vectorizer_path)

print("Spam feature extraction completed successfully!")
